# YouTube Audio Studio · Colab 一键运行

**项目完整内嵌**：Notebook 自带 `studio/` 源码、UI、数据库和测试文件。不需要事先上传 ZIP。

功能：YouTube 单视频 / 频道 / 播放列表 → MP3 → GPU Whisper 或 API → 时间戳文本 → 可选 LLM Prompt 生成文章；5 / 10 / 全部；默认 3 个视频并发；Drive 断点续跑。

**使用前注意**：只处理获得授权的音视频。Colab 虚拟机会销毁，使用 Google Drive 保存数据；`localhost` 指 Colab 虚拟机，不是自己的 Mac/PC。如果选择公开 Gradio 分享链接，请勿向他人分享该链接。不要把 API Key 硬编码到 Notebook 中。

## 1. 挂载 Google Drive
首次运行会请求授权。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. 固定数据路径（多次重启保持不变）
只恢复 `studio_catalog.db`，而非把所有音频复制到本地。旧 `catalog.db` 不会被覆盖。

In [ ]:
import os
from pathlib import Path

ROOT = '/content/drive/MyDrive/zhuweiyi_corpus'
SCRATCH = '/content/zhuweiyi_scratch'
os.environ['STUDIO_ROOT'] = ROOT
os.environ['STUDIO_SCRATCH'] = SCRATCH
Path(ROOT).mkdir(parents=True, exist_ok=True)
Path(SCRATCH).mkdir(parents=True, exist_ok=True)
print('ROOT:', ROOT)
print('SCRATCH:', SCRATCH)

## 3. 解压 Notebook 内置的完整工程源码
这一步不会删除或覆盖 Drive 里的音频、文本、数据库。

In [ ]:
import base64, io, zipfile, sys
from pathlib import Path
SOURCE_ARCHIVE_B64 = """..."""
PROJECT = Path('/content/youtube-audio-studio')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ARCHIVE_B64))) as archive:
    archive.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))
print('源码已解包：', PROJECT)
print('文件数：', len(list(PROJECT.rglob('*'))))

## 4. 安装 yt-dlp + Deno/EJS、Whisper、Gradio
此单元格会安装项目依赖。`faster-whisper==1.2.1` 固定兼容 `av==18.0.0`，避免旧版 `metadata_errors` 问题。已安装的依赖通常不需要每次重新下载，但 Colab 更换虚拟机时需要重新安装。

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

if not shutil.which('ffmpeg'):
    subprocess.run(['apt-get','update','-qq'], check=True)
    subprocess.run(['apt-get','install','-y','-qq','ffmpeg'], check=True)
if not shutil.which('deno'):
    subprocess.run('curl -fsSL https://deno.land/install.sh | sh', shell=True, executable='/bin/bash', check=True)
os.environ['PATH'] = str(Path.home() / '.deno/bin') + os.pathsep + os.environ['PATH']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT)], check=True)
print('安装完成。')

## 5. 运行前诊断
显示 ffmpeg、Deno、PyAV、Whisper 版本与当前 CUDA 情况。

In [ ]:
from studio.preflight import check_environment
for name, value in check_environment():
    print(f'{name:17} {value}')
print('已有 MP3:', len(list((Path(ROOT) / 'media').glob('*.mp3'))))
print('已转写 TXT:', len(list((Path(ROOT) / 'transcripts').glob('*.txt'))))
print('已生成文章:', len(list((Path(ROOT) / 'articles').glob('*.md'))))

## 6. 启动网页控制台
单元格输出中将显示 Gradio 控制台。默认数量 5、并发 3、LLM 关闭，本地 Whisper 将尝试 GPU，CPU 自动使用 `small`。多次启动后使用相同 ROOT，系统会跳过已完成内容。

**安全**：Colab 启动时设置 `share=True`，会创建临时公开可访问的链接；不要公开链接或共享私人密钥。

In [ ]:
from studio.ui import build_app
import os

app = build_app()
app.queue(max_size=10).launch(server_name='0.0.0.0', share=True, inline=True, debug=False)